## Dev setup — run once per kernel

`%reload_ext autoreload` loads the extension on first run and reloads it on
re-runs, so this cell always picks up any change to the autoreload extension.
With `%autoreload 2`, edits to `print_scripts_tree_d/**` hot-reload on the next
cell run — no kernel restart.

Two helpers are defined here:

- **`preview(make, params, **overrides)`** — build a shape from a params
  dataclass (plus ad-hoc overrides), show it, and return it. One-line iteration.
- **`reload_pkg()`** — escape hatch. If `%autoreload` ever goes stale (e.g. it
  swallowed a reload error mid-edit), call this to force-reimport the whole
  package instead of restarting the kernel.

Always access the library via the module qualifier (`shapes.make_*`,
`params.RoundedBoxParams`) so `reload_pkg()` rebinds those names correctly.

In [1]:
# --- Dev setup: run once per kernel ---
%reload_ext autoreload
%autoreload 2

import importlib
import inspect
import logging
import sys
from dataclasses import asdict
from pathlib import Path
from typing import cast

import build123d as bd
from build123d import (
    Box,
    Compound,
    Cone,
    Cylinder,
    Part,
    Pos,
    Rot,
    ShapeList,
    export_stl,
)
from ocp_vscode import (
    get_defaults,
    reset_show,
    set_defaults,
    set_port,
    show,
    show_object,
)

# Access the library via these module qualifiers (shapes.make_*, params.*)
# so reload_pkg() rebinds the names after a force reload.
import print_scripts_tree_d.shapes as shapes
from print_scripts_tree_d import params
from print_scripts_tree_d.export import save_stl

logging.basicConfig(level=logging.INFO, format="%(message)s")
logging.getLogger("build123d").setLevel(logging.WARNING)
models = Path("models")
models.mkdir(exist_ok=True)
reset_show()
set_port(3939)


def reload_pkg(pkg="print_scripts_tree_d"):
    """Force-reimport every loaded submodule of *pkg*, deepest first.

    Escape hatch when %autoreload silently misses a change (e.g. after a
    reload error) — call this instead of restarting the kernel.
    """
    mods = [
        m for n, m in sys.modules.items() if n == pkg or n.startswith(pkg + ".")
    ]
    for m in sorted(mods, key=lambda m: m.__name__.count("."), reverse=True):
        importlib.reload(m)


def preview(make, p=None, **overrides):
    """Build a shape from a params dataclass + overrides, show, and return it.

    Dataclass fields the make_* doesn't accept are dropped; explicit
    overrides are passed straight through (so a typo'd override still raises).
    """
    sig = set(inspect.signature(make).parameters)
    kwargs = {k: v for k, v in (asdict(p) if p else {}).items() if k in sig}
    kwargs.update(overrides)
    shape = make(**kwargs)
    reset_show()
    show(shape)
    return shape

ImportError: cannot import name 'cone' from 'build123d' (c:\Users\alonr.OG-PC\Code\print_scripts_tree_d\.venv\Lib\site-packages\build123d\__init__.py)

## Box with Cylinder Clip

In [ ]:
cp = params.CylinderClipParams()

bp = params.RoundedBoxParams(
    length=90.0,
    width=10.0,
    height=6.0,
    wall_thickness=3.0,
    corner_radius=2.5,
    top_fillet_radius=5.0,
    bottom_fillet_radius=1.0,
)
box = preview(shapes.make_rounded_box, bp)

In [ ]:
clip = preview(
    shapes.make_cylinder_clip,
    cp,
    bore_diameter=11,
    body_depth=9.5,
    tab_width=10.0,
    bore_floor_fillet_r=10,
    tab_length=3.0,
    tab_protrusion=0.4,
    tab_count=2,
    wall_thickness=1.7,
    flat_fillet_r=0.5,
    flat_inner_margin=0.5,
    flat_bottom=True,
    flange_overlap=1
)

## Clip overhang check (FDM, arc fitting)

Clip in print orientation (`Rot(0, 90, 0)`, bed at Z=0). Curved surfaces self-support as **arcs**, so they're judged by geometry rather than a flat 45° cutoff: each circular feature is fit to its radius `R`, and from the layer height we find the near-horizontal cap that's too flat to self-support (per-layer step > `MAX_STEP`). The chord of that cap is the **top bridge span** — a hole prints clean if it's short. Flat (planar) downward faces have no arc and are flagged when their own per-layer step exceeds `MAX_STEP`. The bore renders **gold**, flat overhangs **red**.

In [ ]:
# --- FDM overhang check (arc-fitting model) ---
# Curved surfaces self-support as arcs, so judge them by geometry, not a flat
# 45deg cutoff. For each circular feature we fit the arc (radius R) and, from
# the layer height, find the near-horizontal cap too flat to self-support
# (per-layer step > MAX_STEP); its chord is the top bridge span. A hole is
# fine if that bridge is short. Flat (planar) downward faces have no arc and
# are flagged when their own per-layer step exceeds MAX_STEP.
import math

from build123d import GeomType, Rot

LAYER_H = 0.2      # print layer height (mm)
MAX_STEP = 0.4     # max self-supporting horizontal step per layer (mm)
MAX_BRIDGE = 10.0  # longest clean unsupported bridge (mm)

cp_clip = Rot(0, 90, 0) * clip         # print orientation: bed Z0, build +Z
phi_c = math.atan(LAYER_H / MAX_STEP)  # half-angle of the unsupported cap
zmin = cp_clip.bounding_box().min.Z


def step_per_layer(nz):
    nz = max(-1.0, min(0.0, nz))
    return LAYER_H * (-nz) / math.sqrt(max(1e-12, 1 - nz * nz))


# Cylinders share the horizontal insertion axis through the origin, so a
# face normal pointing toward that axis (negative radial dot) is a bore.
holes, flat_overhang, seen = [], [], {}
for f in cp_clip.faces():
    p = f.position_at(0.5, 0.5)
    n = f.normal_at(p)
    if f.geom_type == GeomType.CYLINDER:
        r = f.radius
        if r is None:  # trimmed/blended cylinder w/o a single defined radius
            continue
        kind = "hole" if (n.Y * p.Y + n.Z * p.Z) < 0 else "boss"
        seen[(round(r, 1), kind)] = 2 * r * math.sin(phi_c)
        if kind == "hole":
            holes.append(f)
    elif f.geom_type == GeomType.PLANE:
        if (
            n.Z < 0
            and step_per_layer(n.Z) > MAX_STEP
            and f.bounding_box().min.Z > zmin + 0.5
        ):
            flat_overhang.append(f)

print("Arc features:")
for (R, kind), span in sorted(seen.items()):
    if kind == "hole":
        verdict = "OK" if span <= MAX_BRIDGE else "NEEDS SUPPORT"
        print(f"  R={R:4.1f} hole -> top bridge {span:4.1f} mm  [{verdict}]")
    else:
        print(f"  R={R:4.1f} boss -> convex, self-supporting")
print(f"{len(flat_overhang)} flat downward face(s) need support")

reset_show()
show(
    cp_clip,
    ShapeList(holes),
    ShapeList(flat_overhang),
    names=["clip", "bore (arc)", "flat overhang"],
    colors=["lightgray", "gold", "red"],
    alphas=[0.55, 0.5, 1.0],
)

In [ ]:
save_stl(clip, str(models / "clip.stl"))

In [ ]:
tx = bp.length / 2 + cp.body_depth / 2
clip_rotated = Rot(0, 90, 0) * clip
# Derive z_offset from the actual clip geometry so it stays correct
# regardless of which flat_inner_margin was used to build the clip.
flat_bottom_z = clip_rotated.bounding_box().min.Z
z_offset = -flat_bottom_z - bp.height / 2
clip_at_end = Pos(tx, 0, z_offset) * clip_rotated

result = box + clip_at_end
if isinstance(result, ShapeList):
    assembly = Compound(children=list(result))
else:
    assembly = cast(Compound, result)

# Lift so both flat faces sit on Z = 0.
assembly = cast(Compound, Pos(0, 0, bp.height / 2) * assembly)

reset_show()
show(assembly)

In [ ]:
save_stl(assembly, str(models / "box_with_clip.stl"))

## Column

In [ ]:
col_body = Cylinder(1.5, 50)
col_foot = Cone(bottom_radius=1, top_radius=1.5, height=3)
column = shapes.make_column(
    body=col_body,
    height=50,
    foot=col_foot,
    diameter=3,
    gusset_size=3,
    gusset_thickness=1.5,
    gusset_position_z="top",
    gusset_orientation_xy=(0,120,240)
)
show(column)

## Table top

In [ ]:
table_top = preview(
    shapes.make_hexagonal_mesh,
    params.HexPanelParams(
        length=6,
        width=6,
        thickness=1,
        hex_radius=4,
        spacing=1,
        outer_border=1,
    ),
)

## Table assembly

In [ ]:
table = shapes.make_table(
    table_top=table_top,
    columns=[column.rotate(angle=180, axis=bd.Axis.X)],
    column_positions=[(50, 50)],
)
reset_show()
show(table)

In [ ]:
save_stl(table, str(models / "table.stl"))

## Rounded Box

In [ ]:
box = preview(
    shapes.make_rounded_box,
    params.RoundedBoxParams(),
    top_fillet_radius=0.8,
    bottom_fillet_radius=0.5,
)

In [ ]:
save_stl(box, str(models / "rounded_box.stl"))

## Magnet ring panel (rounded-corner prism, concave walls)

In [14]:
magnet_panel = preview(
    shapes.make_magnet_ring_panel,
    params.MagnetRingPanelParams(
        outer_diameter=24,
        thickness=4,
        bore_diameter=3,
        bore_top_diameter=8,  # cone bore: 3 mm bottom -> 8 mm top
        magnet_diameter=6,
        magnet_thickness=2.5,
        number_of_magnets=3,
        ring_margin=3,
        wall_concavity=0.4,  # 0 = straight walls; higher = deeper waist
        bore_fillet_radius=0.6,  # round the bore mouth so it isn't sharp
        outer_fillet_radius=0.7,  # round the outside edge; capped near
        #                           (thickness - magnet_thickness) / 2
        pocket_fillet_radius=0.3,  # small edge break at slot floor/ceiling
        clearance=0,  # extra space around the magnet pockets for easier printing and assembly
        top_slot_length=4.0,  # small obround slit cut into the top face above each magnet
        top_slot_width=2.0,  # tangential width of that obround top
    ),
)
save_stl(magnet_panel, str(models / "magnet_ring_panel.stl"))


outer_diameter 24 is too small for the requested magnet layout; panel will be expanded to 33.


+

Exported models\magnet_ring_panel.stl


magnet_ring_panel.stl validated — watertight, volume=1319.4 mm³


In [17]:
# Attach an external part on top of the panel. build123d imports STL natively
# but not OBJ, so load the OBJ with trimesh and hand it over as a mesh shell
# (an import_stl-style Face). It is a triangulated shell, not a clean solid,
# so we assemble it as a separate body in a Compound rather than booleaning it
# into the panel; slicers union overlapping bodies at print time.
import tempfile

import numpy as np
import trimesh


def cylinder_axis_xy(mesh, lo=0.2, hi=0.8, n=9):
    """Estimate a near-cylindrical mesh's central axis (x, y) by a median
    least-squares circle fit over slices through its main body. More robust
    than the bounding-box centre when the part has an asymmetric feature
    (which pulls the bbox off the true axis)."""
    bb = mesh.bounds
    zmin, zmax = bb[0][2], bb[1][2]
    centers = []
    for f in np.linspace(lo, hi, n):
        sec = mesh.section(
            plane_origin=[0, 0, zmin + f * (zmax - zmin)],
            plane_normal=[0, 0, 1],
        )
        if sec is None:
            continue
        x, y = sec.vertices[:, 0], sec.vertices[:, 1]
        a = np.c_[2 * x, 2 * y, np.ones(len(x))]
        cx, cy, _ = np.linalg.lstsq(a, x * x + y * y, rcond=None)[0]
        centers.append((cx, cy))
    centers = np.array(centers)
    return float(np.median(centers[:, 0])), float(np.median(centers[:, 1]))


def to_build123d(mesh):
    """Convert a trimesh mesh to a build123d shape (via a temp STL)."""
    tmp = Path(tempfile.gettempdir()) / "imported_mesh.stl"
    mesh.export(tmp)
    return bd.import_stl(str(tmp))


raw_holder = trimesh.load(
    str(models / "screw_attachmentobj.obj"), force="mesh"
)
# Align the holder's cylinder axis to the bore (origin); nudge to fine-tune.
axis_x, axis_y = cylinder_axis_xy(raw_holder)
nudge_xy = (0.02, -0.14) 
holder = to_build123d(raw_holder)
holder_on_panel = Pos(
    -axis_x + nudge_xy[0],
    -axis_y + nudge_xy[1],
    magnet_panel.bounding_box().max.Z - holder.bounding_box().min.Z,
) * holder

magnet_panel_with_holder = Compound(
    children=[magnet_panel, holder_on_panel]
)
reset_show()
show(
    magnet_panel,
    holder_on_panel,
    names=["magnet panel", "ptfe holder"],
    colors=["lightgray", "steelblue"],
)
export_stl(
    magnet_panel_with_holder,
    str(models / "magnet_panel_with_holder.stl"),
)


cc


True

In [ ]:
washer = preview(
    shapes.make_washer,
    params.WasherParams(
        outer_diameter=9.5,
        hole_diameter=6,
        thickness=4,
    ),
)
save_stl(washer, str(models / "washer.stl"))

In [ ]:
screw_attachment = preview(
    shapes.make_screw_part,
    params.ScrewPartParams(
        outer_diameter=5.0,
        thickness=20.0,
        thread_pitch=7.0,
        bore_diameter=2.0,  # central through-bore -> hollow screw
    ),
)
save_stl(screw_attachment, str(models / "screw_part.stl"))
